# Imports and checking the quality of the data

In [4]:
from PIL import Image
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import shutil
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

def check_images(folder):
    bad = []
    for root, _, files in os.walk(folder):
        for file in files:
            path = os.path.join(root, file)
            try:
                img = Image.open(path)
                img.verify()
            except:
                bad.append(path)
    return bad

bad_images = check_images("data/state-farm-distracted-driver-detection/imgs/train")
print(f"Damaged images: {bad_images}")

print("====== CHECK CLASS DISTRIBUTION =====")

def check_class_distribution(folder):

    base = folder
    counts = {}

    for cls in os.listdir(base):
        counts[cls] = len(os.listdir(os.path.join(base, cls)))
    return counts

print(f"Distribution between classes: {check_class_distribution('data/state-farm-distracted-driver-detection/imgs/train')}")



Damaged images: []
====== CHECK CLASS DISTRIBUTION =====
Distribution between classes: {'c7': 2002, 'c0': 2489, 'c9': 2129, 'c8': 1911, 'c1': 2267, 'c6': 2325, 'c3': 2346, 'c4': 2326, 'c5': 2312, 'c2': 2317}
cpu


# Create a validation dataset by spliting the train dataset

In [ ]:
src = "data/state-farm-distracted-driver-detection/imgs/train"
train_dst = "data/state-farm-distracted-driver-detection/imgs/train"
val_dst = "data/state-farm-distracted-driver-detection/imgs"

split_ratio = 0.8

for cls in os.listdir(src):
    cls_path = os.path.join(src, cls)
    images = os.listdir(cls_path)

    random.shuffle(images)
    split_idx = int(len(images) * split_ratio)

    train_imgs = images[:split_idx]
    val_imgs = images[split_idx:]

    # Train
    for img in train_imgs:
        src_path = os.path.join(cls_path, img)
        dst_path = os.path.join(train_dst, cls, img)

        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy(src_path, dst_path)

    # Val°
    for img in val_imgs:
        src_path = os.path.join(cls_path, img)
        dst_path = os.path.join(val_dst, cls, img)

        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy(src_path, dst_path)

print("Split fertig")

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([transforms.Resize(224), transforms.ToTensor()])

train_dataset = datasets.ImageFolder("data/state-farm-distracted-driver-detection/imgs/train", transform=transform)
val_dataset = datasets.ImageFolder("dataset/val", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

model = models.resnet18(pretrained=True)

model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Validation Accuracy: {accuracy:.4f}")

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/train'